In [1]:
import os
import subprocess
import numpy as np
import pandas as pd
import glob
import shutil
from scipy.stats import pearsonr

In [2]:
colabfold_path = "/home/biocomp/Documents/Jazmin/localcolabfold/.pixi/envs/default/bin/colabfold_batch"
base_dir = os.getcwd()
alphafold_dir = os.path.join(base_dir, "results_alphaFold")

In [3]:
def matrix_comparation(target_protein, sequence):
    prediction = os.path.join(alphafold_dir, f"{sequence}.csv")
    real = os.path.join(base_dir, "real_distances", f"{target_protein}.csv")
    
    if not os.path.exists(real):
        raise FileNotFoundError(f"Real matrix not found: {real}")
    if not os.path.exists(prediction):
        raise FileNotFoundError(f"Predicted matrix not found: {prediction}")

    real_matrix = pd.read_csv(real).to_numpy()
    prediction_matrix = pd.read_csv(prediction).to_numpy()

    if prediction_matrix.shape != real_matrix.shape:
        raise ValueError("Matrices must have the same dimensions.")

    indices_triup = np.triu_indices_from(prediction_matrix, k=1)
    v1 = prediction_matrix[indices_triup]
    v2 = real_matrix[indices_triup]

    drmsd = np.sqrt(np.mean((v1 - v2)**2))
    corr, _ = pearsonr(v1, v2)

    return {
        "dRMSD": drmsd,
        "Pearson_Correlation": corr
    }

In [4]:
def get_representative_coords_from_pdb(pdb_path):
    residues = {}
    residue_order = []

    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM  "):
                atom_name = line[12:16].strip()
                res_name = line[17:20].strip()
                chain_id = line[21]
                res_seq = line[22:27].strip()
                
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())

                res_key = f"{chain_id}_{res_seq}"

                if res_key not in residues:
                    residues[res_key] = {'name': res_name, 'CA': None, 'CB': None}
                    residue_order.append(res_key)

                if atom_name == 'CA':
                    residues[res_key]['CA'] = np.array([x, y, z])
                elif atom_name == 'CB':
                    residues[res_key]['CB'] = np.array([x, y, z])

    coords = []
    for key in residue_order:
        res = residues[key]
        if res['name'] == 'GLY':
            coord = res['CA']
        else:
            coord = res['CB'] if res['CB'] is not None else res['CA']
        
        if coord is not None:
            coords.append(coord)
            
    return np.array(coords)

In [5]:
def get_alphafold_files(sequence):
    os.makedirs(alphafold_dir, exist_ok=True)
    dist_matrix_file = os.path.join(alphafold_dir, f"{sequence}.csv")

    if not os.path.exists(dist_matrix_file):  # !(•̀ᴗ•́)و ̑̑
        fasta_path = os.path.join(base_dir, f"{sequence}.fasta")
        with open(fasta_path, "w") as f:
            f.write(f">{sequence}\n{sequence}")
            
        custom_env = os.environ.copy()
        custom_env["MPLBACKEND"] = "Agg"
        custom_env["TF_CPP_MIN_LOG_LEVEL"] = "0"
        custom_env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
        custom_env["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"
        
        # Plan B de memoria incluido por si acaso
        custom_env["TF_FORCE_UNIFIED_MEMORY"] = "1"
        custom_env["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "4.0"
        
        try:
            # MODO SEGURO: Quitamos "--use-gpu-relax"
            result = subprocess.run([
                colabfold_path, fasta_path, alphafold_dir,
                "--num-recycle", "3" 
            ], check=True, env=custom_env, capture_output=True, text=True)

            # ¡CORRECCIÓN! Todo este bloque ahora está DENTRO del try,
            # lo que significa que se ejecutará si AlphaFold termina exitosamente.
            
            # Verificar que el PDB realmente se creó (buscará archivos rank_001)
            pdb_files = glob.glob(os.path.join(alphafold_dir, f"{sequence}*rank_001*.pdb"))
            if not pdb_files:
                raise FileNotFoundError(f"AlphaFold did not generate a PDB for {sequence}")

            # Procesar el PDB generado
            coords_pred = get_representative_coords_from_pdb(pdb_files[0])
            N = len(coords_pred)
            matrix_pred = np.zeros((N, N))
            for i in range(N):
                for j in range(N):
                    matrix_pred[i, j] = np.linalg.norm(coords_pred[i] - coords_pred[j])
            
            # Guardar la matriz de distancias
            column_names = [f"c{i+1}" for i in range(N)]
            df_matrix = pd.DataFrame(matrix_pred, columns=column_names)
            df_matrix.to_csv(dist_matrix_file, index=False)

        except subprocess.CalledProcessError as e:
            # Esto SOLO se ejecuta si falla
            print(f"Error de ColabFold detectado. Código: {e.returncode}")
            print(f"Error estándar (AQUÍ ESTÁ LA PISTA):\n{e.stderr}")
            raise 

        finally:
            # Limpiar archivos temporales
            for item in glob.glob(os.path.join(alphafold_dir, f"{sequence}*")):
                if not item.endswith(f"{sequence}.csv"):
                    if os.path.isdir(item):
                        shutil.rmtree(item)
                    else:
                        os.remove(item)
            
            for extra in ["cite.bibtex", "config.json", "log.txt"]:
                f_extra = os.path.join(alphafold_dir, extra)
                if os.path.exists(f_extra):
                    os.remove(f_extra)
            if os.path.exists(fasta_path):
                os.remove(fasta_path)

In [18]:
sequence = "RQDACCLFFSLFLLCGWVQC"
get_alphafold_files(sequence)

In [ ]:
target = "1s7m"
resultados = matrix_comparation(target, sequence)
print(f"dRMSD: {resultados['dRMSD']:.4f} Å")
print(f"Correlación: {resultados['Pearson_Correlation']:.4f}")